<a href="https://colab.research.google.com/github/Natanon-k/229352-Statistical-Learning-for-Data-Science-2/blob/main/lab02_data_preprocessing.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

### Statistical Learning for Data Science 2 (229352)
#### Instructor: Donlapark Ponnoprat

#### [Course website](https://donlapark.pages.dev/229352/)

## Lab #2

We will preprocess data, then train them using three tabular black-box models: [TabPFN](https://github.com/PriorLabs/tabpfn), [TabICL](https://github.com/soda-inria/tabicl) and [RealMLP](https://github.com/dholzmueller/pytabkit).

In [1]:
!pip install -q tabpfn tabicl pytabkit

import os
os.environ["TABPFN_MODEL_VERSION"] = "v2"

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, OneHotEncoder, OrdinalEncoder
from sklearn.metrics import root_mean_squared_error

# Black-box models for Exercise 1
from tabpfn import TabPFNRegressor
from tabicl import TabICLRegressor
from pytabkit import RealMLP_TD_Regressor

pd.set_option('display.max_columns', None)
pd.set_option('display.width', 1000)

/usr/local/lib/python3.13/dist-packages/torch/jit/_script.py:1491: FutureWarning: `torch.jit.script` is deprecated. Please switch to `torch.compile` or `torch.export`.
  warnings.warn(


## Part 1: Steel Industry Energy Dataset - Manual Data Preprocessing

### Load the Steel Industry Energy Dataset ([Data Information](https://archive.ics.uci.edu/dataset/851/steel+industry+energy+consumption))

The data is related with the electricity consumption of DAEWOO Steel Co. Ltd in Gwangyang, South Korea, in 2018. The original data has one row every 15 minutes; here, each row is one hour, and we use the last 50 days of 2018. The column `Usage_kWh` is the electricity used in the hour, and `Load_Type` is the load level of the hour (`'Light_Load'`, `'Medium_Load'` or `'Maximum_Load'`).

In [2]:
steel_url = 'https://raw.githubusercontent.com/donlap/ds352-labs/main/steel_industry_hourly.csv'

df = pd.read_csv(steel_url, parse_dates=['date'])
df = df[df['date'] >= '2018-11-12'].copy()  # the last 50 days, so that the models train quickly
df['hour'] = df['date'].dt.hour
print("Shape of the dataset:", df.shape)
df.head()

Shape of the dataset: (1200, 11)


,date,Usage_kWh,Lagging_Current_Reactive.Power_kVarh,Leading_Current_Reactive_Power_kVarh,CO2(tCO2),Lagging_Current_Power_Factor,Leading_Current_Power_Factor,WeekStatus,Day_of_week,Load_Type,hour
7560,2018-11-12 00:00:00,12.31,17.00,0.0,0.0,58.665,100.0,Weekday,Monday,Light_Load,0
7561,2018-11-12 01:00:00,11.22,16.06,0.0,0.0,57.285,100.0,Weekday,Monday,Light_Load,1
7562,2018-11-12 02:00:00,11.16,16.09,0.0,0.0,56.995,100.0,Weekday,Monday,Light_Load,2
7563,2018-11-12 03:00:00,13.56,21.49,0.0,0.0,53.390,100.0,Weekday,Monday,Light_Load,3
7564,2018-11-12 04:00:00,10.96,14.29,0.0,0.0,60.885,100.0,Weekday,Monday,Light_Load,4


### Data Exploration

In [3]:
print("--- Missing Values Count ---")
print(df.isnull().sum())

--- Missing Values Count ---
date                                    0
Usage_kWh                               0
Lagging_Current_Reactive.Power_kVarh    0
Leading_Current_Reactive_Power_kVarh    0
CO2(tCO2)                               0
Lagging_Current_Power_Factor            0
Leading_Current_Power_Factor            0
WeekStatus                              0
Day_of_week                             0
Load_Type                               0
hour                                    0
dtype: int64


In [4]:
print("--- Unique Values for Categorical Columns ---")
for col in df.select_dtypes(include='object').columns:
    print(f"\n'{col}' unique values:")
    print(df[col].value_counts(dropna=False)) # Include NaN counts

--- Unique Values for Categorical Columns ---

'WeekStatus' unique values:
WeekStatus
Weekday    864
Weekend    336
Name: count, dtype: int64

'Day_of_week' unique values:
Day_of_week
Monday       192
Tuesday      168
Wednesday    168
Thursday     168
Friday       168
Saturday     168
Sunday       168
Name: count, dtype: int64

'Load_Type' unique values:
Load_Type
Light_Load      654
Medium_Load     312
Maximum_Load    234
Name: count, dtype: int64


### Data Preprocessing

In [5]:
# Sliding window: the usage of the last 3 hours
df['Usage_lag1'] = df.groupby('Load_Type')['Usage_kWh'].shift(1)
df['Usage_lag2'] = df.groupby('Load_Type')['Usage_kWh'].shift(2)
df['Usage_lag3'] = df.groupby('Load_Type')['Usage_kWh'].shift(3)

# Drop the first 3 rows, which have no past usage
df = df.iloc[3:]

# Drop 'CO2(tCO2)' due to data leakage (it is the usage times a constant)
df = df.drop('CO2(tCO2)', axis=1)

# Define features (X) and target (y = 'Usage_kWh')
X = df.drop('Usage_kWh', axis=1)
y = df['Usage_kWh']

# Split the data BEFORE any transformations (time series: shuffle=False)
X_train,X_test,y_train,y_test = train_test_split(X, y, test_size=0.2, shuffle = False)

# Print data shape
X_train.shape


(957, 12)

We will apply `StandardScaler()`, `OrdinalEncoder()`, `OneHotEncoder()`, cyclical encoding and squaring on a few selected columns.

**1. Numerical Feature: `Usage_lag1` and `Lagging_Current_Reactive.Power_kVarh` (Standard Scaling)**

In [6]:
num_cols_demo = ['Usage_lag1', 'Lagging_Current_Reactive.Power_kVarh']

scaler = StandardScaler()

# Fit the scaler ONLY on the training data
scaler.fit(X_train[num_cols_demo])
X_train_scaled = scaler.transform(X_train[num_cols_demo])
X_test_scaled = scaler.transform(X_test[num_cols_demo])

X_train_scaled

array([[-0.78637058, -0.48658374],
       [-0.76780438, -0.60461515],
       [-0.78791777, -0.63002469],
       ...,
       [-0.70390568, -0.83887471],
       [-0.70204906, -0.63248368],
       [ 0.64485193, -0.67051602]], shape=(957, 2))

**2. Ordinal Feature: `Load_Type` (Ordinal Encoding)**

In [7]:
load_categories = ['Light_Load', 'Medium_Load', 'Maximum_Load']

In [8]:
ord_col_demo = ['Load_Type']

ordinal_encoder = OrdinalEncoder(categories=[load_categories])

## Write your code here
ordinal_encoder.fit(X_train[['Load_Type']])
X_train_ord = ordinal_encoder.transform(X_train[['Load_Type']])
X_test_ord = ordinal_encoder.transform(X_test[['Load_Type']])


**3. Nominal Feature: `Day_of_week` (One-Hot Encoding)**

In [9]:
nom_col_demo = ['Day_of_week']

onehot_encoder = OneHotEncoder(handle_unknown='ignore', sparse_output=False)

## Write your code here
onehot_encoder.fit(X_train[['Day_of_week']])
X_train_onehot = onehot_encoder.transform(X_train[['Day_of_week']])
X_test_onehot = onehot_encoder.transform(X_test[['Day_of_week']])

X_test_onehot

array([[0., 0., 1., ..., 0., 0., 0.],
       [0., 0., 1., ..., 0., 0., 0.],
       [0., 0., 1., ..., 0., 0., 0.],
       ...,
       [0., 1., 0., ..., 0., 0., 0.],
       [0., 1., 0., ..., 0., 0., 0.],
       [0., 1., 0., ..., 0., 0., 0.]], shape=(240, 7))

**4. Cyclical Feature: `hour` (Cyclical Encoding)**

$$
\text{hour} \longmapsto \left(\sin\frac{2\pi \cdot \text{hour}}{24},\ \cos\frac{2\pi \cdot \text{hour}}{24}\right)
$$

In [10]:
cyc_col_demo = 'hour'

## Write your code here
X_train_sin = np.sin(2 * np.pi * X_train[['hour']] / 24)
X_train_cos = np.cos(2 * np.pi * X_train[['hour']] / 24)

X_test_sin = np.sin(2 * np.pi * X_test[['hour']] / 24)
X_test_cos = np.cos(2 * np.pi * X_test[['hour']] / 24)

X_train_sin


,hour
7563,0.707107
7564,0.866025
7565,0.965926
7566,1.000000
7567,0.965926
...,...
8515,-0.965926
8516,-0.866025
8517,-0.707107
8518,-0.500000


**5. Squared Feature: `Lagging_Current_Reactive.Power_kVarh` (Squaring)**

In [11]:
sq_col_demo = 'Lagging_Current_Reactive.Power_kVarh'

## Write your code here
X_train_sq = X_train[['Lagging_Current_Reactive.Power_kVarh']] ** 2
X_test_sq = X_test[['Lagging_Current_Reactive.Power_kVarh']] ** 2

X_train_sq


,Lagging_Current_Reactive.Power_kVarh
7563,461.8201
7564,204.2041
7565,162.3076
7566,150.7984
7567,283.5856
...,...
8515,0.0000
8516,0.0000
8517,0.0000
8518,158.5081


### Train a Black-Box Model

**1. Training and Test Sets with All Preprocessed Features (Standard Scaling)**

In [12]:
# Put the preprocessed features of the training set together, and the same for the test set
## Write your code here
X_train_new = np.column_stack([X_train_scaled, X_train_ord, X_train_onehot, X_train_sin, X_train_cos, X_train_sq])
X_test_new = np.column_stack([X_test_scaled, X_test_ord, X_test_onehot, X_test_sin, X_test_cos, X_test_sq])

# Fit the scaler ONLY on the training data
X_scaler = StandardScaler()
X_scaler.fit(X_train_new)
X_train_new = X_scaler.transform(X_train_new)
X_test_new = X_scaler.transform(X_test_new)

# Print data shape
X_train_new.shape


(957, 13)

**2. Black-Box Model: `TabPFNRegressor` and the RMSE** ([Documentation](https://github.com/PriorLabs/TabPFN))

In [13]:
model_demo = TabPFNRegressor()

## Write your code here
model_demo.fit(X_train_new, y_train)
y_pred = model_demo.predict(X_test_new)
root_mean_squared_error(y_test, y_pred)


tabpfn-v2-regressor.ckpt: reconstructing file:   0%|          |  0.00B / 44.4MB            

tabpfn-v2-regressor.ckpt: downloading bytes:           |  0.00B            

config.json:   0%|          | 0.00/37.0 [00:00<?, ?B/s]

/usr/local/lib/python3.13/dist-packages/tabpfn/validation.py:234: UserWarning: Running on CPU with more than 200 samples may be slow.
Consider using a GPU or the tabpfn-client API: https://github.com/PriorLabs/tabpfn-client
  _validate_num_samples_for_cpu(


5.758999548426363

## Part 2: Bike Sharing Dataset - Preprocessing & Black-Box Models

### **Exercise 1: Preprocess the Bike Sharing Data & Train Three Black-Box Models**

Now, it's your turn. The Bike Sharing dataset ([Data Information](https://archive.ics.uci.edu/dataset/275/bike+sharing+dataset)) contains the number of bikes rented each day (`cnt`) from Capital Bikeshare in Washington, D.C., in 2011 and 2012, together with the weather of each day. The columns `temp`, `hum` and `windspeed` are already scaled to values between 0 and 1 by the authors of the dataset. The column `weather` is one of `'clear'`, `'mist'`, `'light rain/snow'` and `'heavy rain/snow'`.

**Instructions:**

1. Load the data from `https://raw.githubusercontent.com/donlap/ds352-labs/main/bike_sharing_daily.csv` using `pd.read_csv(..., parse_dates=['date'])`. The target is `cnt`.
2. Use a combination of the preprocessing steps in Part 1: sliding window, squaring, ordinal encoding, one-hot encoding, cyclical encoding and standard scaling. The models need numbers, so drop `date`, and encode or drop `weather` and `weekday`.
3. Split the data with `train_test_split(..., test_size=0.2, shuffle=False)`, so that the test set is the last days. Fit the encoders and the scaler on the training set ONLY.
4. Train the three black-box models on your training set. They have the same `fit()` and `predict()` as scikit-learn models.
    *   `TabPFNRegressor()` ([Documentation](https://github.com/PriorLabs/TabPFN))
    *   `TabICLRegressor()` ([Documentation](https://github.com/soda-inria/tabicl))
    *   `RealMLP_TD_Regressor(random_state=0)` ([Documentation](https://github.com/dholzmueller/pytabkit))
5. Calculate the `root_mean_squared_error` ([Documentation](https://scikit-learn.org/stable/modules/generated/sklearn.metrics.root_mean_squared_error.html)) of each model on the test set, and report your results in the markdown block below.


In [14]:
# --- YOUR CODE FOR EXERCISE 1 STARTS HERE ---

bike_url = 'https://raw.githubusercontent.com/donlap/ds352-labs/main/bike_sharing_daily.csv'
bike_df = pd.read_csv(bike_url, parse_dates=['date'])

# 1.
bike_df['cnt_lag1'] = bike_df['cnt'].shift(1)
bike_df['cnt_lag2'] = bike_df['cnt'].shift(2)
bike_df['cnt_lag3'] = bike_df['cnt'].shift(3)

bike_df = bike_df.iloc[3:].reset_index(drop=True)

# 2.
if pd.api.types.is_numeric_dtype(bike_df['weekday']):
    weekday_num = bike_df['weekday']
else:
    day_map = {
        'Sunday': 0, 'Monday': 1, 'Tuesday': 2, 'Wednesday': 3,
        'Thursday': 4, 'Friday': 5, 'Saturday': 6
    }
    weekday_num = bike_df['weekday'].map(day_map)

bike_df['weekday_sin'] = np.sin(2 * np.pi * weekday_num / 7)
bike_df['weekday_cos'] = np.cos(2 * np.pi * weekday_num / 7)

for col in ['temp', 'hum', 'windspeed']:
    if col in bike_df.columns:
        bike_df[f'{col}_sq'] = bike_df[col] ** 2

drop_cols = ['cnt', 'date', 'weekday'] + [c for c in ['casual', 'registered'] if c in bike_df.columns]
X_bike = bike_df.drop(columns=drop_cols)
y_bike = bike_df['cnt']

# 3.
X_train_b, X_test_b, y_train_b, y_test_b = train_test_split(
    X_bike, y_bike, test_size=0.2, shuffle=False
)

weather_categories = ['clear', 'mist', 'light rain/snow', 'heavy rain/snow']
ord_enc_b = OrdinalEncoder(
    categories=[weather_categories],
    handle_unknown='use_encoded_value',
    unknown_value=-1
)
X_train_b = X_train_b.copy()
X_test_b = X_test_b.copy()
X_train_b['weather_ord'] = ord_enc_b.fit_transform(X_train_b[['weather']])
X_test_b['weather_ord'] = ord_enc_b.transform(X_test_b[['weather']])
X_train_b = X_train_b.drop(columns=['weather'])
X_test_b = X_test_b.drop(columns=['weather'])

cat_cols_b = X_train_b.select_dtypes(include=['object', 'category']).columns.tolist()
if cat_cols_b:
    ohe_b = OneHotEncoder(handle_unknown='ignore', sparse_output=False)
    train_ohe = pd.DataFrame(
        ohe_b.fit_transform(X_train_b[cat_cols_b]),
        columns=ohe_b.get_feature_names_out(cat_cols_b),
        index=X_train_b.index
    )
    test_ohe = pd.DataFrame(
        ohe_b.transform(X_test_b[cat_cols_b]),
        columns=ohe_b.get_feature_names_out(cat_cols_b),
        index=X_test_b.index
    )
    X_train_b = pd.concat([X_train_b.drop(columns=cat_cols_b), train_ohe], axis=1)
    X_test_b = pd.concat([X_test_b.drop(columns=cat_cols_b), test_ohe], axis=1)

scaler_b = StandardScaler()
X_train_b_scaled = scaler_b.fit_transform(X_train_b)
X_test_b_scaled = scaler_b.transform(X_test_b)

# 4 & 5.
models = {
    'TabPFNRegressor()': TabPFNRegressor(),
    'TabICLRegressor()': TabICLRegressor(),
    'RealMLP_TD_Regressor(random_state=0)': RealMLP_TD_Regressor(random_state=0),
}

for name, model in models.items():
    model.fit(X_train_b_scaled, y_train_b)
    y_pred_b = model.predict(X_test_b_scaled)
    rmse = root_mean_squared_error(y_test_b, y_pred_b)
    print(f"{name}: {rmse:.4f}")

/usr/local/lib/python3.13/dist-packages/tabpfn/validation.py:234: UserWarning: Running on CPU with more than 200 samples may be slow.
Consider using a GPU or the tabpfn-client API: https://github.com/PriorLabs/tabpfn-client
  _validate_num_samples_for_cpu(


TabPFNRegressor(): 1267.2719
Checkpoint 'tabicl-regressor-v2-20260212.ckpt' not cached.



tabicl-regressor-v2-20260212.ckpt: reconstructing file:   0%|          |  0.00B /  114MB            

tabicl-regressor-v2-20260212.ckpt: downloading bytes:           |  0.00B            

TabICLRegressor(): 1261.2885


/usr/local/lib/python3.13/dist-packages/pytorch_lightning/utilities/_pytree.py:21: `isinstance(treespec, LeafSpec)` is deprecated, use `isinstance(treespec, TreeSpec) and treespec.is_leaf()` instead.


RealMLP_TD_Regressor(random_state=0): 1213.4562


/usr/local/lib/python3.13/dist-packages/pytorch_lightning/utilities/_pytree.py:21: `isinstance(treespec, LeafSpec)` is deprecated, use `isinstance(treespec, TreeSpec) and treespec.is_leaf()` instead.


**Results of Exercise 1** (double-click this block to type your answers)

Preprocessing steps used:

| Model | RMSE on the test set |
|---|---|
| `TabPFNRegressor()` | 1267.2719 |
| `TabICLRegressor()` | 1261.2885 |
| `RealMLP_TD_Regressor(random_state=0)` | 1213.4562 |

The most accurate model (the smallest RMSE):